# ============================================================
# AUDITORÍA CLIMÁTICA
# ============================================================

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

PRECIP_DIR = PROJECT_ROOT / "data" / "raw" / "climate" / "precipitation"

print("Raíz del proyecto:")
print(PROJECT_ROOT)

print("\nCarpeta de precipitación:")
print(PRECIP_DIR)

print("\n¿Existe?")
print(PRECIP_DIR.exists())

print("\nArchivos encontrados:")
precip_files = sorted(PRECIP_DIR.rglob("*.tif"))

print("Total:", len(precip_files))

for file in precip_files[:5]:
    print(file)

Raíz del proyecto:
c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026

Carpeta de precipitación:
c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\precipitation

¿Existe?
True

Archivos encontrados:
Total: 48
c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\precipitation\2022\PREC_2022_01.tif
c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\precipitation\2022\PREC_2022_02.tif
c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\precipitation\2022\PREC_2022_03.tif
c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\precipitation\2022\PREC_2022_04.tif
c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\precipitation\2022\PREC_2022_05.tif


In [2]:
import rasterio

with rasterio.open(precip_files[0]) as src:
    print("Archivo:", precip_files[0].name)
    print("CRS:", src.crs)
    print("Resolución:", src.res)
    print("Dimensiones:", src.width, "x", src.height)
    print("Bandas:", src.count)
    print("NoData:", src.nodata)
    print("Tipo de dato:", src.dtypes[0])
    print("Bounds:", src.bounds)

Archivo: PREC_2022_01.tif
CRS: EPSG:4326
Resolución: (0.05000000074505806, 0.05000000074505806)
Dimensiones: 65 x 72
Bandas: 1
NoData: -9999.0
Tipo de dato: float32
Bounds: BoundingBox(left=-99.94999880716205, bottom=17.79999952018261, right=-96.69999875873327, top=21.39999957382679)


In [3]:
# Verificar consistencia de todos los archivos CHIRPS

crs_set = set()
res_set = set()
shape_set = set()
nodata_set = set()
bounds_set = set()

for file in precip_files:
    with rasterio.open(file) as src:
        crs_set.add(str(src.crs))
        res_set.add(src.res)
        shape_set.add((src.width, src.height))
        nodata_set.add(src.nodata)
        bounds_set.add(tuple(src.bounds))

print("CRS únicos:", crs_set)
print("Resoluciones únicas:", res_set)
print("Dimensiones únicas:", shape_set)
print("NoData únicos:", nodata_set)
print("Bounds únicos:", len(bounds_set))

CRS únicos: {'EPSG:4326'}
Resoluciones únicas: {(0.05000000074505806, 0.05000000074505806)}
Dimensiones únicas: {(65, 72)}
NoData únicos: {-9999.0}
Bounds únicos: 1


In [4]:
# ============================================================
# COBERTURA TEMPORAL CHIRPS
# ============================================================

import re

years_months = []

for file in precip_files:
    match = re.search(r"PREC_(\d{4})_(\d{2})\.tif", file.name)

    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        years_months.append((year, month))

years_months = sorted(years_months)

print("Primer mes:", years_months[0])
print("Último mes:", years_months[-1])

print("\nArchivos por año:")

from collections import Counter

year_counts = Counter(year for year, month in years_months)

for year, count in sorted(year_counts.items()):
    print(year, "→", count, "archivos")

print("\nMeses faltantes:")

expected = {
    (year, month)
    for year in range(2022, 2026)
    for month in range(1, 13)
}

actual = set(years_months)

missing = sorted(expected - actual)

print(missing if missing else "Ninguno")

Primer mes: (2022, 1)
Último mes: (2025, 12)

Archivos por año:
2022 → 12 archivos
2023 → 12 archivos
2024 → 12 archivos
2025 → 12 archivos

Meses faltantes:
Ninguno


In [5]:
# ============================================================
# AUDITORÍA DE VALORES DE PRECIPITACIÓN
# ============================================================

import numpy as np

for file in [precip_files[0], precip_files[-1]]:
    
    with rasterio.open(file) as src:
        data = src.read(1, masked=True)
        
        print("\nArchivo:", file.name)
        print("Mínimo:", data.min())
        print("Máximo:", data.max())
        print("Media:", data.mean())
        print("Mediana:", np.ma.median(data))
        print("Píxeles válidos:", data.count())
        print("Píxeles NoData:", data.size - data.count())


Archivo: PREC_2022_01.tif
Mínimo: 0.0
Máximo: 131.2022
Media: 21.537549597071852
Mediana: 13.16924
Píxeles válidos: 2032
Píxeles NoData: 2648

Archivo: PREC_2025_12.tif
Mínimo: 0.0
Máximo: 152.55312
Media: 24.196921521284448
Mediana: 15.061261
Píxeles válidos: 2032
Píxeles NoData: 2648


In [6]:
# ============================================================
# AUDITORÍA DE VALORES EN LOS 48 MESES
# ============================================================

import pandas as pd
import numpy as np

precip_stats = []

for file in precip_files:

    with rasterio.open(file) as src:
        data = src.read(1, masked=True)

        precip_stats.append({
            "archivo": file.name,
            "min_mm": float(data.min()),
            "max_mm": float(data.max()),
            "media_mm": float(data.mean()),
            "mediana_mm": float(np.ma.median(data)),
            "validos": int(data.count()),
            "nodata": int(data.size - data.count())
        })

precip_stats_df = pd.DataFrame(precip_stats)

display(precip_stats_df)

,archivo,min_mm,max_mm,media_mm,mediana_mm,validos,nodata
0,PREC_2022_01.tif,0.000000,131.202194,21.537550,13.169240,2032,2648
1,PREC_2022_02.tif,0.000000,106.729340,12.222432,6.914905,2032,2648
2,PREC_2022_03.tif,0.000000,40.774372,7.704464,6.669017,2032,2648
3,PREC_2022_04.tif,4.333226,214.369354,32.937185,28.462482,2032,2648
4,PREC_2022_05.tif,6.922159,90.130287,22.636115,19.394558,2032,2648
5,PREC_2022_06.tif,13.837122,647.784546,97.704986,76.089111,2032,2648
6,PREC_2022_07.tif,6.968579,543.476685,90.484490,73.282318,2032,2648
7,PREC_2022_08.tif,22.636246,532.622253,130.841351,120.940369,2032,2648
8,PREC_2022_09.tif,12.330562,803.756653,180.501630,152.132599,2032,2648
9,PREC_2022_10.tif,12.716383,561.832153,98.979900,88.645264,2032,2648


In [7]:
# ============================================================
# COMPATIBILIDAD ESPACIAL: CHIRPS + PARCELAS
# ============================================================

import geopandas as gpd

PARCELS_PATH = PROJECT_ROOT / "data" / "raw" / "parcels" / "Parcelas_Reto_AGC_CONJUNTO.shp"

parcels = gpd.read_file(PARCELS_PATH)

print("CRS parcelas:", parcels.crs)
print("Número de parcelas:", len(parcels))

print("\nCRS CHIRPS:", precip_files[0])

CRS parcelas: EPSG:4326
Número de parcelas: 197

CRS CHIRPS: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\precipitation\2022\PREC_2022_01.tif


In [8]:
with rasterio.open(precip_files[0]) as src:
    chirps_crs = src.crs

print("CRS parcelas:", parcels.crs)
print("CRS CHIRPS:", chirps_crs)
print("\n¿Mismo CRS?", parcels.crs == chirps_crs)

CRS parcelas: EPSG:4326
CRS CHIRPS: EPSG:4326

¿Mismo CRS? True


In [9]:
with rasterio.open(precip_files[0]) as src:
    raster_bounds = src.bounds

    print("\nBounds CHIRPS:")
    print(raster_bounds)

    print("\nBounds parcelas:")
    print(parcels.total_bounds)


Bounds CHIRPS:
BoundingBox(left=-99.94999880716205, bottom=17.79999952018261, right=-96.69999875873327, top=21.39999957382679)

Bounds parcelas:
[-98.612993    19.523825   -98.19669951  19.9843652 ]


In [10]:
from shapely.geometry import box

chirps_box = box(
    raster_bounds.left,
    raster_bounds.bottom,
    raster_bounds.right,
    raster_bounds.top
)

parcelas_dentro = parcels.geometry.intersects(chirps_box)

print("\nParcelas que intersectan CHIRPS:", parcelas_dentro.sum())
print("Parcelas fuera de CHIRPS:", (~parcelas_dentro).sum())


Parcelas que intersectan CHIRPS: 197
Parcelas fuera de CHIRPS: 0


In [11]:
# ============================================================
# COBERTURA VÁLIDA DE CHIRPS POR PARCELA
# ============================================================

from rasterio.mask import mask

coverage_results = []

with rasterio.open(precip_files[0]) as src:

    for _, parcel in parcels.iterrows():

        geometry = [parcel.geometry]

        data, _ = mask(
            src,
            geometry,
            crop=True,
            filled=False,
            all_touched=True
        )

        values = data[0]

        valid_pixels = values.count()
        total_pixels = values.size

        coverage_results.append({
            "ID_POLIGON": parcel["ID_POLIGON"],
            "AREA_HA": parcel["área_ha"],
            "valid_pixels": valid_pixels,
            "total_pixels": total_pixels
        })

coverage_df = pd.DataFrame(coverage_results)

display(coverage_df.head(10))

,ID_POLIGON,AREA_HA,valid_pixels,total_pixels
0,AGC_196,8.57,1,1
1,AGC_044,0.68,1,1
2,AGC_020,8.34,1,1
3,AGC_174,6.89,2,2
4,AGC_083,2.87,1,1
5,AGC_149,8.99,1,1
6,AGC_052,6.23,1,1
7,AGC_009,0.38,1,1
8,AGC_119,0.64,1,1
9,AGC_064,2.45,1,1


In [12]:
print("Parcelas analizadas:", len(coverage_df))
print("Parcelas con 0 píxeles válidos:",
      (coverage_df["valid_pixels"] == 0).sum())

print("\nPíxeles válidos:")
print(coverage_df["valid_pixels"].describe())

Parcelas analizadas: 197
Parcelas con 0 píxeles válidos: 0

Píxeles válidos:
count    197.000000
mean       1.167513
std        0.400714
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        3.000000
Name: valid_pixels, dtype: float64


In [13]:
# ============================================================
# VALIDACIÓN DE NODATA SOBRE LAS PARCELAS
# ============================================================

nodata_by_month = []

for file in precip_files:

    with rasterio.open(file) as src:

        valid_parcels = 0
        invalid_parcels = 0

        for _, parcel in parcels.iterrows():

            data, _ = mask(
                src,
                [parcel.geometry],
                crop=True,
                filled=False,
                all_touched=True
            )

            values = data[0]

            if values.count() > 0:
                valid_parcels += 1
            else:
                invalid_parcels += 1

        nodata_by_month.append({
            "archivo": file.name,
            "parcelas_validas": valid_parcels,
            "parcelas_sin_datos": invalid_parcels
        })

nodata_parcels_df = pd.DataFrame(nodata_by_month)

display(nodata_parcels_df)

,archivo,parcelas_validas,parcelas_sin_datos
0,PREC_2022_01.tif,197,0
1,PREC_2022_02.tif,197,0
2,PREC_2022_03.tif,197,0
3,PREC_2022_04.tif,197,0
4,PREC_2022_05.tif,197,0
5,PREC_2022_06.tif,197,0
6,PREC_2022_07.tif,197,0
7,PREC_2022_08.tif,197,0
8,PREC_2022_09.tif,197,0
9,PREC_2022_10.tif,197,0


In [14]:
print(
    "Meses con alguna parcela sin datos:",
    (nodata_parcels_df["parcelas_sin_datos"] > 0).sum()
)

print(
    "Máximo de parcelas sin datos en un mes:",
    nodata_parcels_df["parcelas_sin_datos"].max()
)

Meses con alguna parcela sin datos: 0
Máximo de parcelas sin datos en un mes: 0


In [15]:
# ============================================================
# COMPARACIÓN DE EXTRACCIÓN CHIRPS
# Enero 2025
# ============================================================

from rasterio.transform import rowcol

test_file = PRECIP_DIR / "2025" / "PREC_2025_01.tif"

results = []

with rasterio.open(test_file) as src:

    # Método A: centroide
    parcels_projected = parcels.to_crs("EPSG:6372")
    centroids_projected = parcels_projected.geometry.centroid
    centroids = centroids_projected.to_crs("EPSG:4326")

    for idx, parcel in parcels.iterrows():

        centroid = centroids.loc[idx]

        # Fila y columna del píxel donde cae el centroide
        row, col = rowcol(src.transform, centroid.x, centroid.y)

        centroid_value = src.read(1, masked=True)[row, col]

        # Método B: todos los píxeles que tocan la parcela
        data, _ = mask(
            src,
            [parcel.geometry],
            crop=True,
            filled=False,
            all_touched=True
        )

        touched_values = data[0]

        mean_touched = touched_values.mean()

        results.append({
            "ID_POLIGON": parcel["ID_POLIGON"],
            "prec_centroid": float(centroid_value),
            "prec_all_touched": float(mean_touched)
        })

comparison_df = pd.DataFrame(results)

display(comparison_df.head(10))

,ID_POLIGON,prec_centroid,prec_all_touched
0,AGC_196,11.715059,11.715059
1,AGC_044,14.340641,14.340641
2,AGC_020,14.340641,14.340641
3,AGC_174,18.443747,20.248158
4,AGC_083,17.910559,17.910559
5,AGC_149,18.093159,18.093159
6,AGC_052,18.093159,18.093159
7,AGC_009,18.093159,18.093159
8,AGC_119,18.093159,18.093159
9,AGC_064,18.093159,18.093159


In [16]:
print("Diferencia absoluta promedio:")

difference = (
    comparison_df["prec_centroid"]
    - comparison_df["prec_all_touched"]
).abs()

print(difference.mean())

print("\nMáxima diferencia:")
print(difference.max())

Diferencia absoluta promedio:
0.2371305652883049

Máxima diferencia:
3.5453853607177734


In [17]:
# ============================================================
# COMPARACIÓN DE MÉTODOS DE EXTRACCIÓN - 48 MESES
# ============================================================

comparison_48 = []

# Centroides calculados correctamente en CRS proyectado
parcels_projected = parcels.to_crs("EPSG:6372")
centroids = parcels_projected.geometry.centroid.to_crs("EPSG:4326")

for file in precip_files:

    # Obtener año y mes del nombre
    match = re.search(r"PREC_(\d{4})_(\d{2})\.tif", file.name)
    year = int(match.group(1))
    month = int(match.group(2))

    with rasterio.open(file) as src:

        raster_data = src.read(1, masked=True)

        for idx, parcel in parcels.iterrows():

            # -------------------------
            # Método A: centroide
            # -------------------------
            centroid = centroids.loc[idx]

            row, col = rowcol(
                src.transform,
                centroid.x,
                centroid.y
            )

            centroid_value = raster_data[row, col]

            # -------------------------
            # Método B: all_touched
            # -------------------------
            data, _ = mask(
                src,
                [parcel.geometry],
                crop=True,
                filled=False,
                all_touched=True
            )

            touched_values = data[0]

            mean_touched = touched_values.mean()

            comparison_48.append({
                "ID_POLIGON": parcel["ID_POLIGON"],
                "year": year,
                "month": month,
                "prec_centroid": float(centroid_value),
                "prec_all_touched": float(mean_touched)
            })

comparison_48_df = pd.DataFrame(comparison_48)

print("Filas generadas:", len(comparison_48_df))

display(comparison_48_df.head(10))

Filas generadas: 9456


,ID_POLIGON,year,month,prec_centroid,prec_all_touched
0,AGC_196,2022,1,6.847781,6.847781
1,AGC_044,2022,1,8.457767,8.457767
2,AGC_020,2022,1,8.457767,8.457767
3,AGC_174,2022,1,19.384502,21.488529
4,AGC_083,2022,1,21.224924,21.224924
5,AGC_149,2022,1,21.254093,21.254093
6,AGC_052,2022,1,21.254093,21.254093
7,AGC_009,2022,1,21.254093,21.254093
8,AGC_119,2022,1,21.254093,21.254093
9,AGC_064,2022,1,21.254093,21.254093


In [18]:
# ============================================================
# DIFERENCIAS ENTRE MÉTODOS
# ============================================================

comparison_48_df["difference"] = (
    comparison_48_df["prec_centroid"]
    - comparison_48_df["prec_all_touched"]
).abs()

print("Diferencia absoluta promedio:",
      comparison_48_df["difference"].mean())

print("Mediana de la diferencia:",
      comparison_48_df["difference"].median())

print("Máxima diferencia:",
      comparison_48_df["difference"].max())

print(
    "Porcentaje de casos idénticos:",
    (
        comparison_48_df["difference"] == 0
    ).mean() * 100,
    "%"
)

Diferencia absoluta promedio: 0.7926829957098055
Mediana de la diferencia: 0.0
Máxima diferencia: 67.518798828125
Porcentaje de casos idénticos: 84.61294416243655 %


In [19]:
# ============================================================
# MAYORES DIFERENCIAS ENTRE MÉTODOS
# ============================================================

top_differences = (
    comparison_48_df
    .sort_values("difference", ascending=False)
    .head(10)
)

display(
    top_differences[
        [
            "ID_POLIGON",
            "year",
            "month",
            "prec_centroid",
            "prec_all_touched",
            "difference"
        ]
    ]
)

,ID_POLIGON,year,month,prec_centroid,prec_all_touched,difference
8841,AGC_145,2025,9,196.586426,264.105225,67.518799
6477,AGC_145,2024,9,257.603882,324.813965,67.210083
4507,AGC_145,2023,11,90.026878,145.683395,55.656517
8842,AGC_123,2025,9,196.586426,250.485494,53.899068
6478,AGC_123,2024,9,257.603882,309.321696,51.717814
8250,AGC_145,2025,6,267.224487,313.941284,46.716797
6307,AGC_174,2024,9,235.744293,279.597839,43.853546
6083,AGC_145,2024,7,272.571747,314.628723,42.056976
4508,AGC_123,2023,11,90.026878,131.342316,41.315437
6461,AGC_048,2024,9,189.566437,230.775238,41.208801


In [20]:
print("Parcelas involucradas en las 10 mayores diferencias:")

print(
    top_differences["ID_POLIGON"].unique()
)

print("\nMeses involucrados:")

print(
    top_differences[["year", "month"]].to_string(index=False)
)

Parcelas involucradas en las 10 mayores diferencias:
<StringArray>
['AGC_145', 'AGC_123', 'AGC_174', 'AGC_048']
Length: 4, dtype: str

Meses involucrados:
 year  month
 2025      9
 2024      9
 2023     11
 2025      9
 2024      9
 2025      6
 2024      9
 2024      7
 2023     11
 2024      9


In [21]:
# ============================================================
# DIAGNÓSTICO DE AGC_145
# ============================================================

parcel_id = "AGC_145"

parcel = parcels[parcels["ID_POLIGON"] == parcel_id].iloc[0]

print("ID:", parcel["ID_POLIGON"])
print("Área:", parcel["área_ha"], "ha")
print("Geometría:", parcel.geometry.geom_type)

# Centroide calculado correctamente en CRS proyectado
parcel_projected = parcels_projected[
    parcels_projected["ID_POLIGON"] == parcel_id
].iloc[0]

centroid = gpd.GeoSeries(
    [parcel_projected.geometry.centroid],
    crs="EPSG:6372"
).to_crs("EPSG:4326").iloc[0]

print("Centroide:", centroid.x, centroid.y)

ID: AGC_145
Área: 8.33 ha
Geometría: Polygon
Centroide: -98.59465025767447 19.550291509434977


In [22]:
test_file = PRECIP_DIR / "2025" / "PREC_2025_09.tif"

with rasterio.open(test_file) as src:

    data, transform = mask(
        src,
        [parcel.geometry],
        crop=True,
        filled=False,
        all_touched=True
    )

    values = data[0]

    print("Píxeles intersectados:", values.count())
    print("Valores de precipitación:")
    print(values.compressed())

Píxeles intersectados: 2
Valores de precipitación:
[196.58643 331.624  ]


In [23]:
with rasterio.open(test_file) as src:

    row, col = rowcol(
        src.transform,
        centroid.x,
        centroid.y
    )

    centroid_value = src.read(1, masked=True)[row, col]

    print("Valor del píxel del centroide:",
          float(centroid_value))

Valor del píxel del centroide: 196.58642578125


In [24]:
# ============================================================
# EXTRACCIÓN PONDERADA POR ÁREA
# Caso de prueba: AGC_145 - septiembre 2025
# ============================================================

parcel_id = "AGC_145"
test_file = PRECIP_DIR / "2025" / "PREC_2025_09.tif"

parcel = parcels[parcels["ID_POLIGON"] == parcel_id].iloc[0]

# Convertimos SOLO la geometría de la parcela a EPSG:6372
parcel_projected = parcels_projected[
    parcels_projected["ID_POLIGON"] == parcel_id
].iloc[0]

with rasterio.open(test_file) as src:

    data, transform = mask(
        src,
        [parcel.geometry],
        crop=True,
        filled=False,
        all_touched=True
    )

    values = data[0]

    results_weighted = []

    rows, cols = np.where(~values.mask)

    for row, col in zip(rows, cols):

        # Geometría del píxel en EPSG:4326
        pixel_left, pixel_top = transform * (col, row)
        pixel_right, pixel_bottom = transform * (col + 1, row + 1)

        pixel_geometry = box(
            pixel_left,
            pixel_bottom,
            pixel_right,
            pixel_top
        )

        # Convertimos el píxel a GeoSeries para reproyectarlo
        pixel_projected = gpd.GeoSeries(
            [pixel_geometry],
            crs="EPSG:4326"
        ).to_crs("EPSG:6372").iloc[0]

        # Intersección en metros
        intersection = (
            parcel_projected.geometry
            .intersection(pixel_projected)
        )

        intersection_area = intersection.area

        if intersection_area > 0:

            results_weighted.append({
                "pixel_row": row,
                "pixel_col": col,
                "precip_mm": float(values[row, col]),
                "intersection_area_m2": intersection_area
            })

weighted_df = pd.DataFrame(results_weighted)

# Peso de cada píxel dentro de la parcela
weighted_df["peso"] = (
    weighted_df["intersection_area_m2"]
    / weighted_df["intersection_area_m2"].sum()
)

# Contribución de cada píxel
weighted_df["contribucion_mm"] = (
    weighted_df["precip_mm"]
    * weighted_df["peso"]
)

display(weighted_df)

,pixel_row,pixel_col,precip_mm,intersection_area_m2,peso,contribucion_mm
0,0,0,196.586426,54942.357392,0.663586,130.451985
1,1,0,331.623993,27853.789001,0.336414,111.562979


In [25]:
precip_weighted = weighted_df["contribucion_mm"].sum()

print("Precipitación ponderada:", precip_weighted, "mm")
print("Suma de pesos:", weighted_df["peso"].sum())

Precipitación ponderada: 242.01496401804383 mm
Suma de pesos: 1.0


In [26]:
# ============================================================
# VALIDACIÓN MATEMÁTICA DEL MÉTODO PONDERADO
# ============================================================

parcel_area_m2 = parcel_projected.geometry.area
intersection_area_m2 = weighted_df["intersection_area_m2"].sum()

print("Área de la parcela:", parcel_area_m2, "m²")
print("Área de intersecciones:", intersection_area_m2, "m²")

print(
    "Diferencia de área:",
    abs(parcel_area_m2 - intersection_area_m2),
    "m²"
)

print(
    "¿Coinciden?",
    np.isclose(parcel_area_m2, intersection_area_m2)
)

Área de la parcela: 82796.14639335373 m²
Área de intersecciones: 82796.14639335312 m²
Diferencia de área: 6.111804395914078e-10 m²
¿Coinciden? True


In [27]:
print("Peso mínimo:", weighted_df["peso"].min())
print("Peso máximo:", weighted_df["peso"].max())
print("Suma de pesos:", weighted_df["peso"].sum())

print(
    "¿Pesos válidos?",
    (
        (weighted_df["peso"] >= 0).all()
        and np.isclose(weighted_df["peso"].sum(), 1.0)
    )
)

Peso mínimo: 0.336414074982132
Peso máximo: 0.663585925017868
Suma de pesos: 1.0
¿Pesos válidos? True


In [28]:
min_precip = weighted_df["precip_mm"].min()
max_precip = weighted_df["precip_mm"].max()

print("Precipitación mínima:", min_precip, "mm")
print("Precipitación máxima:", max_precip, "mm")
print("Precipitación ponderada:", precip_weighted, "mm")

print(
    "¿Resultado dentro del rango?",
    min_precip <= precip_weighted <= max_precip
)

Precipitación mínima: 196.58642578125 mm
Precipitación máxima: 331.6239929199219 mm
Precipitación ponderada: 242.01496401804383 mm
¿Resultado dentro del rango? True


In [29]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from shapely.geometry import box

PROJECT_ROOT = Path.cwd().parent

PARCELS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "parcels"
    / "Parcelas_Reto_AGC_CONJUNTO.shp"
)

PRECIP_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "climate"
    / "precipitation"
    / "2025"
    / "PREC_2025_09.tif"
)

parcels = gpd.read_file(PARCELS_PATH)

with rasterio.open(PRECIP_PATH) as src:
    precip_crs = src.crs
    precip_transform = src.transform
    precip_data = src.read(1)
    precip_nodata = src.nodata
    precip_bounds = src.bounds

print("Parcelas:", len(parcels))
print("CRS parcelas:", parcels.crs)
print("CRS raster:", precip_crs)
print("Raster:", src.width, "x", src.height)
print("NoData:", precip_nodata)
print("Bounds raster:", precip_bounds)

Parcelas: 197
CRS parcelas: EPSG:4326
CRS raster: EPSG:4326
Raster: 65 x 72
NoData: -9999.0
Bounds raster: BoundingBox(left=-99.94999880716205, bottom=17.79999952018261, right=-96.69999875873327, top=21.39999957382679)


In [30]:
# Seleccionamos una parcela concreta para validar la geometría
parcel_id = "AGC_145"

parcel = parcels.loc[parcels["ID_POLIGON"] == parcel_id].copy()

print("Número de parcelas encontradas:", len(parcel))
print("ID:", parcel.iloc[0]["ID_POLIGON"])
print("Área reportada (ha):", parcel.iloc[0]["área_ha"])
print("CRS:", parcel.crs)
print("Geometría:", parcel.iloc[0].geometry.geom_type)
print("Área geométrica en EPSG:4326:", parcel.iloc[0].geometry.area)


Número de parcelas encontradas: 1
ID: AGC_145
Área reportada (ha): 8.33
CRS: EPSG:4326
Geometría: Polygon
Área geométrica en EPSG:4326: 7.171106999995098e-06


In [31]:
# Reproyectamos temporalmente la parcela a un CRS métrico
parcel_metric = parcel.to_crs("EPSG:6372")

area_m2 = parcel_metric.iloc[0].geometry.area
area_ha_calculada = area_m2 / 10_000

print("Área geométrica en m²:", area_m2)
print("Área geométrica en ha:", area_ha_calculada)
print("Área reportada en ha:", parcel.iloc[0]["área_ha"])

diferencia_ha = area_ha_calculada - parcel.iloc[0]["área_ha"]
diferencia_pct = (diferencia_ha / parcel.iloc[0]["área_ha"]) * 100

print("Diferencia absoluta (ha):", diferencia_ha)
print("Diferencia relativa (%):", diferencia_pct)

Área geométrica en m²: 82796.14639335373
Área geométrica en ha: 8.279614639335373
Área reportada en ha: 8.33
Diferencia absoluta (ha): -0.0503853606646274
Diferencia relativa (%): -0.6048662744853229


In [32]:
# Reproyectamos todas las parcelas a un CRS métrico
parcels_metric = parcels.to_crs("EPSG:6372").copy()

# Área geométrica calculada a partir del polígono
parcels_metric["area_m2_calculada"] = parcels_metric.geometry.area
parcels_metric["area_ha_calculada"] = (
    parcels_metric["area_m2_calculada"] / 10_000
)

# Diferencia respecto al área reportada
parcels_metric["diferencia_ha"] = (
    parcels_metric["area_ha_calculada"] - parcels_metric["área_ha"]
)

parcels_metric["diferencia_pct"] = (
    parcels_metric["diferencia_ha"]
    / parcels_metric["área_ha"]
    * 100
)

print("Parcelas analizadas:", len(parcels_metric))
print()
print("Diferencia porcentual:")
print(parcels_metric["diferencia_pct"].describe())

Parcelas analizadas: 197

Diferencia porcentual:
count    197.000000
mean      -0.680865
std        0.173971
min       -1.936400
25%       -0.725241
50%       -0.675148
75%       -0.619360
max       -0.163290
Name: diferencia_pct, dtype: float64


In [33]:
# Revisamos las parcelas con mayor diferencia absoluta porcentual
revision_areas = (
    parcels_metric[
        [
            "ID_POLIGON",
            "área_ha",
            "area_ha_calculada",
            "diferencia_ha",
            "diferencia_pct",
        ]
    ]
    .assign(
        diferencia_abs_pct=lambda df: df["diferencia_pct"].abs()
    )
    .sort_values("diferencia_abs_pct", ascending=False)
)

revision_areas.head(10)

,ID_POLIGON,área_ha,area_ha_calculada,diferencia_ha,diferencia_pct,diferencia_abs_pct
146,AGC_069,0.38,0.372642,-0.007358,-1.936400,1.936400
7,AGC_009,0.38,0.373148,-0.006852,-1.803192,1.803192
44,AGC_089,1.72,1.703006,-0.016994,-0.988008,0.988008
139,AGC_049,0.87,0.861481,-0.008519,-0.979221,0.979221
91,AGC_057,0.63,0.623875,-0.006125,-0.972281,0.972281
102,AGC_032,1.44,1.426015,-0.013985,-0.971183,0.971183
112,AGC_017,1.60,1.584627,-0.015373,-0.960785,0.960785
118,AGC_076,1.01,1.000362,-0.009638,-0.954247,0.954247
125,AGC_139,1.92,1.902315,-0.017685,-0.921083,0.921083
134,AGC_175,1.41,1.397053,-0.012947,-0.918199,0.918199


In [34]:
from rasterio.transform import xy
from shapely.geometry import box

# Parcela de prueba en CRS métrico
parcel_geom = parcel_metric.iloc[0].geometry

# Transformamos el raster de CHIRPS a geometrías de píxel
with rasterio.open(PRECIP_PATH) as src:
    transform = src.transform
    width = src.width
    height = src.height

    pixel_intersections = []

    for row in range(height):
        for col in range(width):

            x_left, y_top = xy(
                transform,
                row,
                col,
                offset="ul"
            )

            x_right, y_bottom = xy(
                transform,
                row,
                col,
                offset="lr"
            )

            pixel = box(
                x_left,
                y_bottom,
                x_right,
                y_top
            )

            if parcel_geom.intersects(
                gpd.GeoSeries([pixel], crs="EPSG:4326")
                .to_crs("EPSG:6372")
                .iloc[0]
            ):
                pixel_intersections.append(
                    (row, col, pixel)
                )

print("Píxeles que intersectan AGC_145:", len(pixel_intersections))

Píxeles que intersectan AGC_145: 2


In [35]:
# Calculamos el área de intersección de cada píxel con AGC_145
intersections = []

for row, col, pixel_wgs84 in pixel_intersections:

    # Pasamos el píxel a CRS métrico
    pixel_metric = (
        gpd.GeoSeries([pixel_wgs84], crs="EPSG:4326")
        .to_crs("EPSG:6372")
        .iloc[0]
    )

    intersection = parcel_geom.intersection(pixel_metric)

    intersections.append({
        "row": row,
        "col": col,
        "intersection_area_m2": intersection.area
    })

intersections_df = pd.DataFrame(intersections)

print(intersections_df)
print()
print(
    "Área total de intersecciones:",
    intersections_df["intersection_area_m2"].sum(),
    "m²"
)
print(
    "Área de la parcela:",
    parcel_geom.area,
    "m²"
)
print(
    "Diferencia:",
    intersections_df["intersection_area_m2"].sum() - parcel_geom.area,
    "m²"
)

   row  col  intersection_area_m2
0   36   27          54942.357392
1   37   27          27853.789001

Área total de intersecciones: 82796.14639335312 m²
Área de la parcela: 82796.14639335373 m²
Diferencia: -6.111804395914078e-10 m²


In [36]:
# Calculamos el peso de cada píxel dentro de la parcela
intersections_df["peso"] = (
    intersections_df["intersection_area_m2"]
    / parcel_geom.area
)

print(intersections_df)
print()

print("Suma de pesos:", intersections_df["peso"].sum())
print("Error respecto a 1:", intersections_df["peso"].sum() - 1)

   row  col  intersection_area_m2      peso
0   36   27          54942.357392  0.663586
1   37   27          27853.789001  0.336414

Suma de pesos: 0.9999999999999927
Error respecto a 1: -7.327471962526033e-15


In [37]:
# Extraemos el valor de precipitación de cada píxel
with rasterio.open(PRECIP_PATH) as src:
    values = []

    for row, col in zip(
        intersections_df["row"],
        intersections_df["col"]
    ):
        value = src.read(1)[row, col]
        values.append(value)

intersections_df["precipitacion_mm"] = values

print(intersections_df)

   row  col  intersection_area_m2      peso  precipitacion_mm
0   36   27          54942.357392  0.663586        196.586426
1   37   27          27853.789001  0.336414        331.623993


In [38]:
# Calculamos la precipitación ponderada por área
intersections_df["contribucion_mm"] = (
    intersections_df["precipitacion_mm"]
    * intersections_df["peso"]
)

precipitacion_area_weighted = (
    intersections_df["contribucion_mm"].sum()
)

print("Precipitación area-weighted:", precipitacion_area_weighted, "mm")
print()
print("Mínimo de píxeles:", intersections_df["precipitacion_mm"].min(), "mm")
print("Máximo de píxeles:", intersections_df["precipitacion_mm"].max(), "mm")

Precipitación area-weighted: 242.01496401804206 mm

Mínimo de píxeles: 196.58643 mm
Máximo de píxeles: 331.624 mm


In [39]:
comparacion_agc145 = pd.DataFrame({
    "metodo": [
        "centroid",
        "all_touched",
        "area_weighted"
    ],
    "precipitacion_mm": [
        196.58642578125,
        264.105224609375,
        242.01496401804206
    ]
})

comparacion_agc145

,metodo,precipitacion_mm
0,centroid,196.586426
1,all_touched,264.105225
2,area_weighted,242.014964


In [40]:
with rasterio.open(PRECIP_PATH) as src:
    raster = src.read(1)

    for row, col in zip(
        intersections_df["row"],
        intersections_df["col"]
    ):
        value = raster[row, col]

        print(
            f"Pixel ({row}, {col}) -> "
            f"{value} mm | "
            f"NoData: {value == src.nodata}"
        )

Pixel (36, 27) -> 196.58642578125 mm | NoData: False
Pixel (37, 27) -> 331.6239929199219 mm | NoData: False


In [41]:
# Todas las parcelas en CRS métrico para cálculos de área
parcels_metric = parcels.to_crs("EPSG:6372").copy()

print("Parcelas:", len(parcels_metric))
print("CRS:", parcels_metric.crs)
print("Geometrías inválidas:", (~parcels_metric.geometry.is_valid).sum())
print("Geometrías vacías:", parcels_metric.geometry.is_empty.sum())

Parcelas: 197
CRS: EPSG:6372
Geometrías inválidas: 0
Geometrías vacías: 0


In [42]:
with rasterio.open(PRECIP_PATH) as src:
    raster = src.read(1)
    transform = src.transform
    nodata = src.nodata
    width = src.width
    height = src.height

    resultados = []

    # Parcelas originales para centroides
    parcels_wgs84 = parcels.copy()

    for idx in range(len(parcels_metric)):

        parcel_id = parcels_metric.iloc[idx]["ID_POLIGON"]
        geom_metric = parcels_metric.iloc[idx].geometry

        # --------------------------------------------------
        # 1. CENTROID
        # --------------------------------------------------
        centroid_metric = geom_metric.centroid

        centroid_wgs84 = gpd.GeoSeries(
            [centroid_metric],
            crs="EPSG:6372"
        ).to_crs("EPSG:4326").iloc[0]

        centroid_value = next(
            src.sample([(centroid_wgs84.x, centroid_wgs84.y)])
        )[0]

        # --------------------------------------------------
        # 2. PIXELES QUE INTERSECTAN LA PARCELA
        # --------------------------------------------------
        pixel_data = []

        for row in range(height):
            for col in range(width):

                x_left, y_top = xy(
                    transform,
                    row,
                    col,
                    offset="ul"
                )

                x_right, y_bottom = xy(
                    transform,
                    row,
                    col,
                    offset="lr"
                )

                pixel_wgs84 = box(
                    x_left,
                    y_bottom,
                    x_right,
                    y_top
                )

                pixel_metric = gpd.GeoSeries(
                    [pixel_wgs84],
                    crs="EPSG:4326"
                ).to_crs("EPSG:6372").iloc[0]

                if geom_metric.intersects(pixel_metric):

                    value = raster[row, col]

                    # Ignorar NoData
                    if value != nodata:

                        intersection = geom_metric.intersection(
                            pixel_metric
                        )

                        area_m2 = intersection.area

                        if area_m2 > 0:
                            pixel_data.append({
                                "row": row,
                                "col": col,
                                "value": float(value),
                                "area_m2": area_m2
                            })

        # --------------------------------------------------
        # 3. VALIDACION DE PIXELES
        # --------------------------------------------------
        if len(pixel_data) == 0:
            raise ValueError(
                f"{parcel_id}: no se encontraron píxeles válidos."
            )

        pixel_df = pd.DataFrame(pixel_data)

        # --------------------------------------------------
        # 4. ALL TOUCHED
        # --------------------------------------------------
        all_touched_value = pixel_df["value"].mean()

        # --------------------------------------------------
        # 5. AREA WEIGHTED
        # --------------------------------------------------
        total_intersection_area = pixel_df["area_m2"].sum()

        pixel_df["peso"] = (
            pixel_df["area_m2"]
            / total_intersection_area
        )

        area_weighted_value = (
            pixel_df["value"] * pixel_df["peso"]
        ).sum()

        # --------------------------------------------------
        # 6. GUARDAMOS RESULTADOS
        # --------------------------------------------------
        resultados.append({
            "ID_POLIGON": parcel_id,
            "centroid_mm": float(centroid_value),
            "all_touched_mm": float(all_touched_value),
            "area_weighted_mm": float(area_weighted_value),
            "n_pixeles": len(pixel_df),
            "suma_pesos": float(pixel_df["peso"].sum()),
            "area_intersecciones_m2": float(
                total_intersection_area
            ),
            "area_parcela_m2": float(
                geom_metric.area
            )
        })

septiembre_2025 = pd.DataFrame(resultados)

print("Filas obtenidas:", len(septiembre_2025))
septiembre_2025.head()

Filas obtenidas: 197


,ID_POLIGON,centroid_mm,all_touched_mm,area_weighted_mm,n_pixeles,suma_pesos,area_intersecciones_m2,area_parcela_m2
0,AGC_196,153.162109,153.162109,153.162109,1,1.0,85130.668788,85130.668788
1,AGC_044,157.464645,157.464645,157.464645,1,1.0,6760.402971,6760.402971
2,AGC_020,157.464645,157.464645,157.464645,1,1.0,82856.226598,82856.226598
3,AGC_174,151.940598,151.940598,151.940598,1,1.0,68423.862739,68423.862739
4,AGC_083,147.459686,147.459686,147.459686,1,1.0,28466.739798,28466.739798


In [43]:
septiembre_2025.head(20)
#septiembre_2025

,ID_POLIGON,centroid_mm,all_touched_mm,area_weighted_mm,n_pixeles,suma_pesos,area_intersecciones_m2,area_parcela_m2
0,AGC_196,153.162109,153.162109,153.162109,1,1.0,85130.668788,85130.668788
1,AGC_044,157.464645,157.464645,157.464645,1,1.0,6760.402971,6760.402971
2,AGC_020,157.464645,157.464645,157.464645,1,1.0,82856.226598,82856.226598
3,AGC_174,151.940598,151.940598,151.940598,1,1.0,68423.862739,68423.862739
4,AGC_083,147.459686,147.459686,147.459686,1,1.0,28466.739798,28466.739798
5,AGC_149,168.990997,168.990997,168.990997,1,1.0,89310.839000,89310.839000
6,AGC_052,168.990997,168.990997,168.990997,1,1.0,61849.129845,61849.129845
7,AGC_009,168.990997,168.990997,168.990997,1,1.0,3731.478713,3731.478713
8,AGC_119,168.990997,168.990997,168.990997,1,1.0,6378.785496,6378.785496
9,AGC_064,168.990997,168.990997,168.990997,1,1.0,24291.955747,24291.955747


In [44]:
septiembre_2025[
    septiembre_2025["ID_POLIGON"] == "AGC_145"
]

,ID_POLIGON,centroid_mm,all_touched_mm,area_weighted_mm,n_pixeles,suma_pesos,area_intersecciones_m2,area_parcela_m2
173,AGC_145,196.586426,264.105209,242.014964,2,1.0,82796.146393,82796.146393


In [45]:
from shapely.strtree import STRtree

with rasterio.open(PRECIP_PATH) as src:
    raster = src.read(1)
    transform = src.transform
    nodata = src.nodata
    width = src.width
    height = src.height

# --------------------------------------------------
# Construimos las geometrías de TODOS los píxeles
# una sola vez.
# --------------------------------------------------

pixel_geometries = []
pixel_indices = []

for row in range(height):
    for col in range(width):

        x_left, y_top = xy(
            transform,
            row,
            col,
            offset="ul"
        )

        x_right, y_bottom = xy(
            transform,
            row,
            col,
            offset="lr"
        )

        pixel = box(
            x_left,
            y_bottom,
            x_right,
            y_top
        )

        pixel_geometries.append(pixel)
        pixel_indices.append((row, col))

# --------------------------------------------------
# Índice espacial
# --------------------------------------------------

pixel_tree = STRtree(pixel_geometries)

print("Píxeles construidos:", len(pixel_geometries))
print("Índice espacial creado:", pixel_tree is not None)

Píxeles construidos: 4680
Índice espacial creado: True


In [46]:
# --------------------------------------------------
# CONSTRUIR PÍXELES EN CRS MÉTRICO
# --------------------------------------------------

pixel_geometries_metric = []
pixel_indices = []

# Creamos un GeoDataFrame con los 4,680 píxeles
pixel_gdf = gpd.GeoDataFrame(
    {
        "row": [],
        "col": [],
        "geometry": []
    },
    geometry="geometry",
    crs="EPSG:4326"
)

pixel_rows = []
pixel_cols = []
pixel_geometries_wgs84 = []

with rasterio.open(PRECIP_PATH) as src:

    transform = src.transform
    width = src.width
    height = src.height

    for row in range(height):
        for col in range(width):

            x_left, y_top = xy(
                transform,
                row,
                col,
                offset="ul"
            )

            x_right, y_bottom = xy(
                transform,
                row,
                col,
                offset="lr"
            )

            pixel = box(
                x_left,
                y_bottom,
                x_right,
                y_top
            )

            pixel_rows.append(row)
            pixel_cols.append(col)
            pixel_geometries_wgs84.append(pixel)

pixel_gdf = gpd.GeoDataFrame(
    {
        "row": pixel_rows,
        "col": pixel_cols
    },
    geometry=pixel_geometries_wgs84,
    crs="EPSG:4326"
)

# Reproyección UNA sola vez
pixel_gdf_metric = pixel_gdf.to_crs("EPSG:6372")

# Construimos el índice espacial
pixel_tree_metric = STRtree(
    pixel_gdf_metric.geometry.values
)

print("Píxeles:", len(pixel_gdf_metric))
print("CRS:", pixel_gdf_metric.crs)
print("Índice espacial creado:", pixel_tree_metric is not None)

Píxeles: 4680
CRS: EPSG:6372
Índice espacial creado: True


In [47]:
print(
    "Geometrías inválidas:",
    (~pixel_gdf_metric.geometry.is_valid).sum()
)

print(
    "Geometrías vacías:",
    pixel_gdf_metric.geometry.is_empty.sum()
)

print(
    "Geometrías nulas:",
    pixel_gdf_metric.geometry.isna().sum()
)

Geometrías inválidas: 0
Geometrías vacías: 0
Geometrías nulas: 0


In [48]:
# --------------------------------------------------
# PRUEBA DEL EXTRACTOR OPTIMIZADO
# SOLO AGC_145
# --------------------------------------------------

parcel_id = "AGC_145"

parcel_row = parcels_metric[
    parcels_metric["ID_POLIGON"] == parcel_id
].iloc[0]

geom_metric = parcel_row.geometry

# --------------------------------------------------
# 1. CENTROID
# --------------------------------------------------

centroid_metric = geom_metric.centroid

centroid_wgs84 = gpd.GeoSeries(
    [centroid_metric],
    crs="EPSG:6372"
).to_crs("EPSG:4326").iloc[0]

with rasterio.open(PRECIP_PATH) as src:
    centroid_value = float(
        next(
            src.sample(
                [(centroid_wgs84.x, centroid_wgs84.y)]
            )
        )[0]
    )

# --------------------------------------------------
# 2. CANDIDATOS DEL ÍNDICE ESPACIAL
# --------------------------------------------------

candidate_indices = pixel_tree_metric.query(
    geom_metric
)

print("Píxeles candidatos:", len(candidate_indices))

# --------------------------------------------------
# 3. INTERSECCIONES REALES
# --------------------------------------------------

pixel_data = []

for candidate_idx in candidate_indices:

    pixel = pixel_gdf_metric.geometry.iloc[candidate_idx]

    if not geom_metric.intersects(pixel):
        continue

    row = int(
        pixel_gdf_metric.iloc[candidate_idx]["row"]
    )

    col = int(
        pixel_gdf_metric.iloc[candidate_idx]["col"]
    )

    value = float(raster[row, col])

    if value == nodata:
        continue

    intersection = geom_metric.intersection(pixel)

    area_m2 = intersection.area

    if area_m2 > 0:

        pixel_data.append({
            "row": row,
            "col": col,
            "value": value,
            "area_m2": area_m2
        })

pixel_df_test = pd.DataFrame(pixel_data)

print("Píxeles con intersección válida:")
print(len(pixel_df_test))

# --------------------------------------------------
# 4. ALL TOUCHED
# --------------------------------------------------

all_touched_value = pixel_df_test["value"].mean()

# --------------------------------------------------
# 5. AREA WEIGHTED
# --------------------------------------------------

total_area = pixel_df_test["area_m2"].sum()

pixel_df_test["peso"] = (
    pixel_df_test["area_m2"] / total_area
)

area_weighted_value = (
    pixel_df_test["value"]
    * pixel_df_test["peso"]
).sum()

print()
print("RESULTADOS OPTIMIZADOS")
print("----------------------")
print("Centroid:", centroid_value)
print("All touched:", all_touched_value)
print("Area weighted:", area_weighted_value)
print("Suma pesos:", pixel_df_test["peso"].sum())
print("Área intersecciones:", total_area)
print("Área parcela:", geom_metric.area)

Píxeles candidatos: 2
Píxeles con intersección válida:
2

RESULTADOS OPTIMIZADOS
----------------------
Centroid: 196.58642578125
All touched: 264.10520935058594
Area weighted: 242.01496401804383
Suma pesos: 1.0
Área intersecciones: 82796.14639335312
Área parcela: 82796.14639335373


In [49]:
# --------------------------------------------------
# PRUEBA CORREGIDA DEL EXTRACTOR OPTIMIZADO
# SOLO AGC_145
# --------------------------------------------------

with rasterio.open(PRECIP_PATH) as src:
    raster = src.read(1)
    transform = src.transform
    nodata = src.nodata

parcel_id = "AGC_145"

parcel_row = parcels_metric[
    parcels_metric["ID_POLIGON"] == parcel_id
].iloc[0]

geom_metric = parcel_row.geometry

# --------------------------------------------------
# 1. CENTROID
# --------------------------------------------------

centroid_metric = geom_metric.centroid

centroid_wgs84 = gpd.GeoSeries(
    [centroid_metric],
    crs="EPSG:6372"
).to_crs("EPSG:4326").iloc[0]

# Convertimos coordenadas -> fila/columna del raster
row_centroid, col_centroid = rasterio.transform.rowcol(
    transform,
    centroid_wgs84.x,
    centroid_wgs84.y
)

centroid_value = float(
    raster[row_centroid, col_centroid]
)

# --------------------------------------------------
# 2. CANDIDATOS
# --------------------------------------------------

candidate_indices = pixel_tree_metric.query(
    geom_metric
)

pixel_data = []

for candidate_idx in candidate_indices:

    pixel = pixel_gdf_metric.geometry.iloc[candidate_idx]

    if not geom_metric.intersects(pixel):
        continue

    row = int(
        pixel_gdf_metric.iloc[candidate_idx]["row"]
    )

    col = int(
        pixel_gdf_metric.iloc[candidate_idx]["col"]
    )

    value = float(raster[row, col])

    if value == nodata:
        continue

    intersection = geom_metric.intersection(pixel)

    area_m2 = intersection.area

    if area_m2 > 0:

        pixel_data.append({
            "row": row,
            "col": col,
            "value": value,
            "area_m2": area_m2
        })

pixel_df_test = pd.DataFrame(pixel_data)

# --------------------------------------------------
# 3. ALL TOUCHED
# --------------------------------------------------

all_touched_value = pixel_df_test["value"].mean()

# --------------------------------------------------
# 4. AREA WEIGHTED
# --------------------------------------------------

total_area = pixel_df_test["area_m2"].sum()

pixel_df_test["peso"] = (
    pixel_df_test["area_m2"] / total_area
)

area_weighted_value = (
    pixel_df_test["value"]
    * pixel_df_test["peso"]
).sum()

print("Píxeles candidatos:", len(candidate_indices))
print("Píxeles válidos:", len(pixel_df_test))
print()
print("Centroid:", centroid_value)
print("All touched:", all_touched_value)
print("Area weighted:", area_weighted_value)
print("Suma pesos:", pixel_df_test["peso"].sum())
print("Área intersecciones:", total_area)
print("Área parcela:", geom_metric.area)

Píxeles candidatos: 2
Píxeles válidos: 2

Centroid: 196.58642578125
All touched: 264.10520935058594
Area weighted: 242.01496401804383
Suma pesos: 1.0
Área intersecciones: 82796.14639335312
Área parcela: 82796.14639335373


In [50]:
import time

inicio = time.perf_counter()

# --------------------------------------------------
# EXTRACCIÓN OPTIMIZADA
# 197 PARCELAS - SEPTIEMBRE 2025
# --------------------------------------------------

resultados_optimizado = []

with rasterio.open(PRECIP_PATH) as src:
    raster = src.read(1)
    transform = src.transform
    nodata = src.nodata

    for idx in range(len(parcels_metric)):

        parcel_id = parcels_metric.iloc[idx]["ID_POLIGON"]
        geom_metric = parcels_metric.iloc[idx].geometry

        # --------------------------------------------------
        # 1. CENTROID
        # --------------------------------------------------

        centroid_metric = geom_metric.centroid

        centroid_wgs84 = gpd.GeoSeries(
            [centroid_metric],
            crs="EPSG:6372"
        ).to_crs("EPSG:4326").iloc[0]

        row_centroid, col_centroid = rasterio.transform.rowcol(
            transform,
            centroid_wgs84.x,
            centroid_wgs84.y
        )

        centroid_value = float(
            raster[row_centroid, col_centroid]
        )

        # --------------------------------------------------
        # 2. CANDIDATOS ESPACIALES
        # --------------------------------------------------

        candidate_indices = pixel_tree_metric.query(
            geom_metric
        )

        pixel_data = []

        for candidate_idx in candidate_indices:

            pixel = pixel_gdf_metric.geometry.iloc[candidate_idx]

            if not geom_metric.intersects(pixel):
                continue

            row = int(
                pixel_gdf_metric.iloc[candidate_idx]["row"]
            )

            col = int(
                pixel_gdf_metric.iloc[candidate_idx]["col"]
            )

            value = float(raster[row, col])

            # NoData
            if value == nodata:
                continue

            intersection = geom_metric.intersection(pixel)

            area_m2 = intersection.area

            if area_m2 > 0:

                pixel_data.append({
                    "row": row,
                    "col": col,
                    "value": value,
                    "area_m2": area_m2
                })

        # --------------------------------------------------
        # 3. VALIDACIÓN
        # --------------------------------------------------

        if len(pixel_data) == 0:
            raise ValueError(
                f"{parcel_id}: no se encontraron píxeles válidos."
            )

        pixel_df = pd.DataFrame(pixel_data)

        # --------------------------------------------------
        # 4. ALL TOUCHED
        # --------------------------------------------------

        all_touched_value = pixel_df["value"].mean()

        # --------------------------------------------------
        # 5. AREA WEIGHTED
        # --------------------------------------------------

        total_area = pixel_df["area_m2"].sum()

        pixel_df["peso"] = (
            pixel_df["area_m2"]
            / total_area
        )

        area_weighted_value = (
            pixel_df["value"]
            * pixel_df["peso"]
        ).sum()

        # --------------------------------------------------
        # 6. RESULTADO
        # --------------------------------------------------

        resultados_optimizado.append({
            "ID_POLIGON": parcel_id,
            "centroid_mm": centroid_value,
            "all_touched_mm": float(all_touched_value),
            "area_weighted_mm": float(area_weighted_value),
            "n_pixeles": len(pixel_df),
            "suma_pesos": float(pixel_df["peso"].sum()),
            "area_intersecciones_m2": float(total_area),
            "area_parcela_m2": float(geom_metric.area)
        })

septiembre_2025_optimizado = pd.DataFrame(
    resultados_optimizado
)

fin = time.perf_counter()

print("Filas obtenidas:", len(septiembre_2025_optimizado))
print(f"Tiempo de ejecución: {fin - inicio:.2f} segundos")

Filas obtenidas: 197
Tiempo de ejecución: 0.37 segundos


In [51]:
# --------------------------------------------------
# COMPARACIÓN:
# MÉTODO ORIGINAL vs MÉTODO OPTIMIZADO
# --------------------------------------------------

comparacion = septiembre_2025.merge(
    septiembre_2025_optimizado,
    on="ID_POLIGON",
    suffixes=("_original", "_optimizado")
)

# Diferencias absolutas
comparacion["diff_centroid"] = (
    comparacion["centroid_mm_original"]
    - comparacion["centroid_mm_optimizado"]
).abs()

comparacion["diff_all_touched"] = (
    comparacion["all_touched_mm_original"]
    - comparacion["all_touched_mm_optimizado"]
).abs()

comparacion["diff_area_weighted"] = (
    comparacion["area_weighted_mm_original"]
    - comparacion["area_weighted_mm_optimizado"]
).abs()

print("Parcelas comparadas:", len(comparacion))
print()

print("Diferencia máxima - centroid:",
      comparacion["diff_centroid"].max())

print("Diferencia máxima - all touched:",
      comparacion["diff_all_touched"].max())

print("Diferencia máxima - area weighted:",
      comparacion["diff_area_weighted"].max())

Parcelas comparadas: 197

Diferencia máxima - centroid: 0.0
Diferencia máxima - all touched: 0.0
Diferencia máxima - area weighted: 5.684341886080802e-14


In [52]:
# --------------------------------------------------
# VALIDACIÓN DE INTEGRIDAD
# SEPTIEMBRE 2025 - 197 PARCELAS
# --------------------------------------------------

df = septiembre_2025_optimizado.copy()

# 1. IDs
ids_unicos = df["ID_POLIGON"].nunique()
ids_nulos = df["ID_POLIGON"].isna().sum()

# 2. Valores faltantes
faltantes_centroid = df["centroid_mm"].isna().sum()
faltantes_all = df["all_touched_mm"].isna().sum()
faltantes_weighted = df["area_weighted_mm"].isna().sum()

# 3. Pesos
error_pesos = (df["suma_pesos"] - 1).abs()

# 4. Áreas
error_areas = (
    df["area_intersecciones_m2"]
    - df["area_parcela_m2"]
).abs()

# 5. Valores negativos
neg_centroid = (df["centroid_mm"] < 0).sum()
neg_all = (df["all_touched_mm"] < 0).sum()
neg_weighted = (df["area_weighted_mm"] < 0).sum()

print("Parcelas:", len(df))
print("IDs únicos:", ids_unicos)
print("IDs nulos:", ids_nulos)
print()

print("NaN centroid:", faltantes_centroid)
print("NaN all touched:", faltantes_all)
print("NaN area weighted:", faltantes_weighted)
print()

print("Máximo error suma de pesos:", error_pesos.max())
print("Máximo error de área (m²):", error_areas.max())
print()

print("Precipitaciones negativas:")
print("  Centroid:", neg_centroid)
print("  All touched:", neg_all)
print("  Area weighted:", neg_weighted)

Parcelas: 197
IDs únicos: 197
IDs nulos: 0

NaN centroid: 0
NaN all touched: 0
NaN area weighted: 0

Máximo error suma de pesos: 1.1102230246251565e-16
Máximo error de área (m²): 4.7904904931783676e-08

Precipitaciones negativas:
  Centroid: 0
  All touched: 0
  Area weighted: 0


In [53]:
comparacion_metodos = septiembre_2025_optimizado.copy()

comparacion_metodos["diff_centroid_weighted"] = (
    comparacion_metodos["centroid_mm"]
    - comparacion_metodos["area_weighted_mm"]
).abs()

comparacion_metodos["diff_all_weighted"] = (
    comparacion_metodos["all_touched_mm"]
    - comparacion_metodos["area_weighted_mm"]
).abs()

comparacion_metodos["diff_centroid_all"] = (
    comparacion_metodos["centroid_mm"]
    - comparacion_metodos["all_touched_mm"]
).abs()

print(
    comparacion_metodos[
        [
            "diff_centroid_weighted",
            "diff_all_weighted",
            "diff_centroid_all"
        ]
    ].describe()
)

       diff_centroid_weighted  diff_all_weighted  diff_centroid_all
count              197.000000         197.000000         197.000000
mean                 1.160147           1.328928           2.489076
std                  4.699483           4.612935           8.465573
min                  0.000000           0.000000           0.000000
25%                  0.000000           0.000000           0.000000
50%                  0.000000           0.000000           0.000000
75%                  0.000000           0.000000           0.000000
max                 45.428538          35.020087          67.518784


In [54]:
comparacion_metodos[
    [
        "ID_POLIGON",
        "centroid_mm",
        "all_touched_mm",
        "area_weighted_mm",
        "n_pixeles",
        "diff_centroid_weighted",
        "diff_all_weighted",
        "diff_centroid_all"
    ]
].sort_values(
    "diff_centroid_weighted",
    ascending=False
).head(10)

,ID_POLIGON,centroid_mm,all_touched_mm,area_weighted_mm,n_pixeles,diff_centroid_weighted,diff_all_weighted,diff_centroid_all
173,AGC_145,196.586426,264.105209,242.014964,2,45.428538,22.090245,67.518784
177,AGC_183,140.882797,168.734612,166.218473,2,25.335676,2.516139,27.851814
184,AGC_141,196.586426,168.734612,176.210436,2,20.375990,7.475824,27.851814
174,AGC_123,196.586426,250.485494,215.465407,3,18.878981,35.020087,53.899068
157,AGC_048,140.882797,177.433495,157.775212,2,16.892415,19.658283,36.550697
133,AGC_026,116.888588,132.174137,129.722667,2,12.834079,2.451471,15.285549
182,AGC_163,140.882797,168.734612,152.430392,2,11.547595,16.304220,27.851814
37,AGC_042,258.995483,234.082245,247.912468,2,11.083015,13.830224,24.913239
38,AGC_194,258.995483,234.082245,248.017960,2,10.977524,13.935715,24.913239
126,AGC_133,116.888588,132.174137,127.312466,2,10.423878,4.861671,15.285549


In [55]:
distribucion_pixeles = (
    septiembre_2025_optimizado["n_pixeles"]
    .value_counts()
    .sort_index()
)

print(distribucion_pixeles)

n_pixeles
1    167
2     28
3      2
Name: count, dtype: int64


In [56]:
df = septiembre_2025_optimizado.copy()

df["diff_centroid_weighted_abs"] = (
    df["centroid_mm"] - df["area_weighted_mm"]
).abs()

df["diff_centroid_weighted_pct"] = (
    df["diff_centroid_weighted_abs"]
    / df["area_weighted_mm"]
    * 100
)

df["diff_all_weighted_abs"] = (
    df["all_touched_mm"] - df["area_weighted_mm"]
).abs()

df["diff_all_weighted_pct"] = (
    df["diff_all_weighted_abs"]
    / df["area_weighted_mm"]
    * 100
)

print(
    df[
        [
            "diff_centroid_weighted_pct",
            "diff_all_weighted_pct"
        ]
    ].describe()
)

       diff_centroid_weighted_pct  diff_all_weighted_pct
count                  197.000000             197.000000
mean                     0.674444               0.780003
std                      2.462624               2.559145
min                      0.000000               0.000000
25%                      0.000000               0.000000
50%                      0.000000               0.000000
75%                      0.000000               0.000000
max                     18.770963              16.253230


In [57]:
df = septiembre_2025_optimizado.copy()

df["diff_centroid_weighted_pct"] = (
    (df["centroid_mm"] - df["area_weighted_mm"]).abs()
    / df["area_weighted_mm"]
    * 100
)

df["diff_all_weighted_pct"] = (
    (df["all_touched_mm"] - df["area_weighted_mm"]).abs()
    / df["area_weighted_mm"]
    * 100
)

top_diferencias = (
    df[
        [
            "ID_POLIGON",
            "centroid_mm",
            "all_touched_mm",
            "area_weighted_mm",
            "n_pixeles",
            "diff_centroid_weighted_pct",
            "diff_all_weighted_pct"
        ]
    ]
    .sort_values(
        "diff_centroid_weighted_pct",
        ascending=False
    )
    .head(10)
)

top_diferencias

,ID_POLIGON,centroid_mm,all_touched_mm,area_weighted_mm,n_pixeles,diff_centroid_weighted_pct,diff_all_weighted_pct
173,AGC_145,196.586426,264.105209,242.014964,2,18.770963,9.127636
177,AGC_183,140.882797,168.734612,166.218473,2,15.242395,1.513754
184,AGC_141,196.586426,168.734612,176.210436,2,11.563441,4.242555
157,AGC_048,140.882797,177.433495,157.775212,2,10.706634,12.459678
133,AGC_026,116.888588,132.174137,129.722667,2,9.893474,1.889778
174,AGC_123,196.586426,250.485494,215.465407,3,8.761954,16.253230
126,AGC_133,116.888588,132.174137,127.312466,2,8.187634,3.818692
182,AGC_163,140.882797,168.734612,152.430392,2,7.575651,10.696174
81,AGC_033,116.888588,123.285450,122.906115,2,4.896036,0.308638
20,AGC_059,123.569923,132.176434,129.790710,2,4.792936,1.838132


In [58]:
meses_validacion = [
    (2023, 11),
    (2024, 7),
    (2024, 9),
    (2025, 6),
    (2025, 9),
]

for year, month in meses_validacion:
    archivo = PRECIP_DIR / str(year) / f"PREC_{year}_{month:02d}.tif"
    print(
        f"{year}-{month:02d}:",
        archivo.exists(),
        archivo.name
    )

2023-11: True PREC_2023_11.tif
2024-07: True PREC_2024_07.tif
2024-09: True PREC_2024_09.tif
2025-06: True PREC_2025_06.tif
2025-09: True PREC_2025_09.tif


In [60]:
[name for name in globals() if "extra" in name.lower() or "precip" in name.lower()]

['PRECIP_DIR',
 'precip_files',
 'precip_stats',
 'precip_stats_df',
 'precip_weighted',
 'min_precip',
 'max_precip',
 'PRECIP_PATH',
 'precip_crs',
 'precip_transform',
 'precip_data',
 'precip_nodata',
 'precip_bounds',
 'precipitacion_area_weighted']

In [61]:
type(precipitacion_area_weighted)

numpy.float64

In [62]:
print(precipitacion_area_weighted)

242.01496401804206


In [63]:
type(septiembre_2025_optimizado)

pandas.DataFrame

In [64]:
septiembre_2025_optimizado.columns.tolist()

['ID_POLIGON',
 'centroid_mm',
 'all_touched_mm',
 'area_weighted_mm',
 'n_pixeles',
 'suma_pesos',
 'area_intersecciones_m2',
 'area_parcela_m2']

In [65]:
[name for name in globals() if any(
    palabra in name.lower()
    for palabra in [
        "pixel",
        "tree",
        "parcel",
        "geometry",
        "metric"
    ]
)]

['PARCELS_PATH',
 'parcels',
 'parcelas_dentro',
 'parcel',
 'geometry',
 'valid_pixels',
 'total_pixels',
 'valid_parcels',
 'invalid_parcels',
 'nodata_parcels_df',
 'parcels_projected',
 'parcel_id',
 'parcel_projected',
 'pixel_left',
 'pixel_top',
 'pixel_right',
 'pixel_bottom',
 'pixel_geometry',
 'pixel_projected',
 'parcel_area_m2',
 'parcel_metric',
 'parcels_metric',
 'parcel_geom',
 'pixel_intersections',
 'pixel',
 'pixel_wgs84',
 'pixel_metric',
 'parcels_wgs84',
 'geom_metric',
 'centroid_metric',
 'pixel_data',
 'pixel_df',
 'STRtree',
 'pixel_geometries',
 'pixel_indices',
 'pixel_tree',
 'pixel_geometries_metric',
 'pixel_gdf',
 'pixel_rows',
 'pixel_cols',
 'pixel_geometries_wgs84',
 'pixel_gdf_metric',
 'pixel_tree_metric',
 'parcel_row',
 'pixel_df_test',
 'distribucion_pixeles']

In [67]:
import numpy as np
import pandas as pd
import rasterio


def extraer_chirps_area_weighted(
    raster_path,
    parcels_metric,
    pixel_tree_metric,
    pixel_geometries_metric
):
    """
    Extrae precipitación CHIRPS por parcela mediante ponderación por área.

    La parcela y los píxeles deben estar en un CRS métrico.
    """

    resultados = []

    with rasterio.open(raster_path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform

        for _, parcel_row in parcels_metric.iterrows():

            parcel_id = parcel_row["ID_POLIGON"]
            parcel_geom = parcel_row.geometry
            parcel_area_m2 = parcel_geom.area

            # Píxeles candidatos mediante STRtree
            candidate_indices = pixel_tree_metric.query(parcel_geom)

            pixel_intersections = []

            for idx in candidate_indices:

                pixel_geom = pixel_geometries_metric[idx]

                if not parcel_geom.intersects(pixel_geom):
                    continue

                intersection = parcel_geom.intersection(pixel_geom)

                if intersection.is_empty or intersection.area <= 0:
                    continue

                # Obtener fila/columna del píxel
                row = pixel_rows[idx]
                col = pixel_cols[idx]

                value = data[row, col]

                # Ignorar NoData
                if nodata is not None and np.isclose(value, nodata):
                    continue

                if not np.isfinite(value):
                    continue

                intersection_area = intersection.area

                pixel_intersections.append(
                    {
                        "pixel_index": idx,
                        "precip_mm": float(value),
                        "intersection_area_m2": intersection_area
                    }
                )

            if len(pixel_intersections) == 0:
                raise ValueError(
                    f"{parcel_id}: no se encontraron píxeles válidos"
                )

            intersections_df = pd.DataFrame(pixel_intersections)

            # Pesos espaciales
            intersections_df["peso"] = (
                intersections_df["intersection_area_m2"]
                / parcel_area_m2
            )

            # Precipitación ponderada
            area_weighted_mm = (
                intersections_df["precip_mm"]
                * intersections_df["peso"]
            ).sum()

            resultados.append(
                {
                    "ID_POLIGON": parcel_id,
                    "area_weighted_mm": float(area_weighted_mm),
                    "n_pixeles": len(intersections_df),
                    "suma_pesos": float(
                        intersections_df["peso"].sum()
                    ),
                    "area_intersecciones_m2": float(
                        intersections_df["intersection_area_m2"].sum()
                    ),
                    "area_parcela_m2": float(parcel_area_m2)
                }
            )

    return pd.DataFrame(resultados)

In [69]:
test_query = pixel_tree_metric.query(
    parcels_metric.geometry.iloc[0]
)

print("Tipo:", type(test_query))
print("Cantidad:", len(test_query))
print("Primeros valores:", test_query[:10])

print(
    "Cantidad de geometrías:",
    len(pixel_geometries_metric)
)

Tipo: <class 'numpy.ndarray'>
Cantidad: 1
Primeros valores: [2237]
Cantidad de geometrías: 0


In [70]:
print(
    "Máximo índice devuelto:",
    test_query.max()
)

print(
    "Longitud pixel_geometries_metric:",
    len(pixel_geometries_metric)
)

Máximo índice devuelto: 2237
Longitud pixel_geometries_metric: 0


In [71]:
import numpy as np
import geopandas as gpd
from shapely.geometry import box
from shapely.strtree import STRtree
import rasterio

raster_path = PRECIP_DIR / "2025" / "PREC_2025_09.tif"

with rasterio.open(raster_path) as src:
    transform = src.transform
    height = src.height
    width = src.width
    raster_crs = src.crs

pixel_geometries_wgs84 = []
pixel_rows = []
pixel_cols = []

for row in range(height):
    for col in range(width):

        left, top = transform * (col, row)
        right, bottom = transform * (col + 1, row + 1)

        geom = box(
            left,
            bottom,
            right,
            top
        )

        pixel_geometries_wgs84.append(geom)
        pixel_rows.append(row)
        pixel_cols.append(col)

pixel_gdf = gpd.GeoDataFrame(
    {
        "row": pixel_rows,
        "col": pixel_cols
    },
    geometry=pixel_geometries_wgs84,
    crs=raster_crs
)

pixel_gdf_metric = pixel_gdf.to_crs(
    parcels_metric.crs
)

pixel_geometries_metric = list(
    pixel_gdf_metric.geometry
)

pixel_tree_metric = STRtree(
    pixel_geometries_metric
)

print("CRS raster:", raster_crs)
print("CRS métrico:", pixel_gdf_metric.crs)
print("Número de píxeles:", len(pixel_geometries_metric))
print("Geometrías vacías:", sum(
    geom.is_empty for geom in pixel_geometries_metric
))

CRS raster: EPSG:4326
CRS métrico: EPSG:6372
Número de píxeles: 4680
Geometrías vacías: 0


In [72]:
test_query = pixel_tree_metric.query(
    parcels_metric.geometry.iloc[0]
)

print("Cantidad de candidatos:", len(test_query))
print("Primeros índices:", test_query[:10])
print("Máximo índice:", test_query.max())
print("Total geometrías:", len(pixel_geometries_metric))

Cantidad de candidatos: 1
Primeros índices: [2237]
Máximo índice: 2237
Total geometrías: 4680


In [73]:
test_sep_2025 = extraer_chirps_area_weighted(
    raster_path=PRECIP_DIR / "2025" / "PREC_2025_09.tif",
    parcels_metric=parcels_metric,
    pixel_tree_metric=pixel_tree_metric,
    pixel_geometries_metric=pixel_geometries_metric
)

print("Filas:", len(test_sep_2025))
print("IDs únicos:", test_sep_2025["ID_POLIGON"].nunique())
print("Nulos:")
print(test_sep_2025.isna().sum())

Filas: 197
IDs únicos: 197
Nulos:
ID_POLIGON                0
area_weighted_mm          0
n_pixeles                 0
suma_pesos                0
area_intersecciones_m2    0
area_parcela_m2           0
dtype: int64


In [74]:
comparacion = septiembre_2025_optimizado[
    [
        "ID_POLIGON",
        "area_weighted_mm"
    ]
].merge(
    test_sep_2025[
        [
            "ID_POLIGON",
            "area_weighted_mm"
        ]
    ],
    on="ID_POLIGON",
    suffixes=("_referencia", "_funcion")
)

comparacion["diferencia_abs"] = (
    comparacion["area_weighted_mm_referencia"]
    - comparacion["area_weighted_mm_funcion"]
).abs()

print(
    "Máxima diferencia:",
    comparacion["diferencia_abs"].max()
)

print(
    "Diferencia promedio:",
    comparacion["diferencia_abs"].mean()
)

Máxima diferencia: 1.5833734323678073e-10
Diferencia promedio: 4.1024645609556e-12


In [75]:
meses_validacion = [
    (2023, 11),
    (2024, 7),
    (2024, 9),
    (2025, 6),
    (2025, 9),
]

resultados_validacion = []

for year, month in meses_validacion:

    print(f"Procesando {year}-{month:02d}...")

    raster_path = (
        PRECIP_DIR
        / str(year)
        / f"PREC_{year}_{month:02d}.tif"
    )

    resultado = extraer_chirps_area_weighted(
        raster_path=raster_path,
        parcels_metric=parcels_metric,
        pixel_tree_metric=pixel_tree_metric,
        pixel_geometries_metric=pixel_geometries_metric
    )

    resultado["year"] = year
    resultado["month"] = month

    resultados_validacion.append(resultado)

    print(
        f"  Parcelas: {len(resultado)} | "
        f"IDs únicos: {resultado['ID_POLIGON'].nunique()} | "
        f"Nulos: {resultado.isna().sum().sum()}"
    )

print("\nValidación terminada.")

Procesando 2023-11...
  Parcelas: 197 | IDs únicos: 197 | Nulos: 0
Procesando 2024-07...
  Parcelas: 197 | IDs únicos: 197 | Nulos: 0
Procesando 2024-09...
  Parcelas: 197 | IDs únicos: 197 | Nulos: 0
Procesando 2025-06...
  Parcelas: 197 | IDs únicos: 197 | Nulos: 0
Procesando 2025-09...
  Parcelas: 197 | IDs únicos: 197 | Nulos: 0

Validación terminada.


In [76]:
validacion_5_meses = pd.concat(
    resultados_validacion,
    ignore_index=True
)

validacion_5_meses["error_pesos"] = (
    validacion_5_meses["suma_pesos"] - 1
).abs()

validacion_5_meses["error_area_m2"] = (
    validacion_5_meses["area_intersecciones_m2"]
    - validacion_5_meses["area_parcela_m2"]
).abs()

print("Filas totales:", len(validacion_5_meses))
print(
    "Máximo error suma de pesos:",
    validacion_5_meses["error_pesos"].max()
)
print(
    "Máximo error de área (m²):",
    validacion_5_meses["error_area_m2"].max()
)
print(
    "Promedio error de área (m²):",
    validacion_5_meses["error_area_m2"].mean()
)

Filas totales: 985
Máximo error suma de pesos: 6.38378239159465e-13
Máximo error de área (m²): 4.7904904931783676e-08
Promedio error de área (m²): 1.543620261505511e-09


In [77]:
print(
    validacion_5_meses[
        [
            "year",
            "month",
            "n_pixeles",
            "suma_pesos",
            "error_pesos",
            "error_area_m2"
        ]
    ].describe()
)

              year       month   n_pixeles    suma_pesos   error_pesos  \
count   985.000000  985.000000  985.000000  9.850000e+02  9.850000e+02   
mean   2024.200000    8.400000    1.162437  1.000000e+00  2.238706e-14   
std       0.748712    1.744445    0.395619  8.798989e-14  8.510545e-14   
min    2023.000000    6.000000    1.000000  1.000000e+00  0.000000e+00   
25%    2024.000000    7.000000    1.000000  1.000000e+00  0.000000e+00   
50%    2024.000000    9.000000    1.000000  1.000000e+00  2.220446e-16   
75%    2025.000000    9.000000    1.000000  1.000000e+00  4.440892e-16   
max    2025.000000   11.000000    3.000000  1.000000e+00  6.383782e-13   

       error_area_m2  
count   9.850000e+02  
mean    1.543620e-09  
std     6.302267e-09  
min     0.000000e+00  
25%     0.000000e+00  
50%     1.273293e-11  
75%     2.910383e-11  
max     4.790490e-08  


In [78]:
# ============================================================
# CHIRPS — EXTRACCIÓN AREA-WEIGHTED COMPLETA
# 2022-01 → 2025-12
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Obtener todos los archivos CHIRPS
# ------------------------------------------------------------

precip_files = sorted(
    PRECIP_DIR.rglob("PREC_*.tif")
)

print("Archivos encontrados:", len(precip_files))

assert len(precip_files) == 48, (
    f"Esperábamos 48 archivos y encontramos {len(precip_files)}"
)

# ------------------------------------------------------------
# 2. Procesar cada mes
# ------------------------------------------------------------

resultados_chirps = []

for i, raster_path in enumerate(precip_files, start=1):

    print(
        f"[{i:02d}/48] "
        f"Procesando {raster_path.name}..."
    )

    resultado = extraer_chirps_area_weighted(
        raster_path=raster_path,
        parcels_metric=parcels_metric,
        pixel_tree_metric=pixel_tree_metric,
        pixel_geometries_metric=pixel_geometries_metric
    )

    # Extraer año y mes desde el nombre:
    # PREC_2025_09.tif
    year = int(raster_path.stem.split("_")[1])
    month = int(raster_path.stem.split("_")[2])

    resultado["year"] = year
    resultado["month"] = month

    resultados_chirps.append(resultado)

# ------------------------------------------------------------
# 3. Concatenar
# ------------------------------------------------------------

chirps_parcel_month = pd.concat(
    resultados_chirps,
    ignore_index=True
)

# Orden lógico
chirps_parcel_month = chirps_parcel_month.sort_values(
    ["ID_POLIGON", "year", "month"]
).reset_index(drop=True)

print("\nExtracción terminada.")
print("Filas:", len(chirps_parcel_month))
print(
    "Parcelas:",
    chirps_parcel_month["ID_POLIGON"].nunique()
)
print(
    "Meses:",
    chirps_parcel_month[["year", "month"]]
    .drop_duplicates()
    .shape[0]
)

Archivos encontrados: 48
[01/48] Procesando PREC_2022_01.tif...
[02/48] Procesando PREC_2022_02.tif...
[03/48] Procesando PREC_2022_03.tif...
[04/48] Procesando PREC_2022_04.tif...
[05/48] Procesando PREC_2022_05.tif...
[06/48] Procesando PREC_2022_06.tif...
[07/48] Procesando PREC_2022_07.tif...
[08/48] Procesando PREC_2022_08.tif...
[09/48] Procesando PREC_2022_09.tif...
[10/48] Procesando PREC_2022_10.tif...
[11/48] Procesando PREC_2022_11.tif...
[12/48] Procesando PREC_2022_12.tif...
[13/48] Procesando PREC_2023_01.tif...
[14/48] Procesando PREC_2023_02.tif...
[15/48] Procesando PREC_2023_03.tif...
[16/48] Procesando PREC_2023_04.tif...
[17/48] Procesando PREC_2023_05.tif...
[18/48] Procesando PREC_2023_06.tif...
[19/48] Procesando PREC_2023_07.tif...
[20/48] Procesando PREC_2023_08.tif...
[21/48] Procesando PREC_2023_09.tif...
[22/48] Procesando PREC_2023_10.tif...
[23/48] Procesando PREC_2023_11.tif...
[24/48] Procesando PREC_2023_12.tif...
[25/48] Procesando PREC_2024_01.tif...


In [79]:
print("Shape:", chirps_parcel_month.shape)

print("\nColumnas:")
print(chirps_parcel_month.columns.tolist())

print("\nDuplicados completos:")
print(chirps_parcel_month.duplicated().sum())

print("\nDuplicados parcela-mes:")
print(
    chirps_parcel_month
    .duplicated(subset=["ID_POLIGON", "year", "month"])
    .sum()
)

print("\nValores nulos:")
print(chirps_parcel_month.isna().sum())

Shape: (9456, 8)

Columnas:
['ID_POLIGON', 'area_weighted_mm', 'n_pixeles', 'suma_pesos', 'area_intersecciones_m2', 'area_parcela_m2', 'year', 'month']

Duplicados completos:
0

Duplicados parcela-mes:
0

Valores nulos:
ID_POLIGON                0
area_weighted_mm          0
n_pixeles                 0
suma_pesos                0
area_intersecciones_m2    0
area_parcela_m2           0
year                      0
month                     0
dtype: int64


In [80]:
cobertura_parcelas = (
    chirps_parcel_month
    .groupby("ID_POLIGON")
    .size()
)

print(cobertura_parcelas.describe())

print(
    "\nParcelas con número de meses distinto de 48:",
    (cobertura_parcelas != 48).sum()
)

print(
    "\nMínimo de meses por parcela:",
    cobertura_parcelas.min()
)

print(
    "Máximo de meses por parcela:",
    cobertura_parcelas.max()
)

count    197.0
mean      48.0
std        0.0
min       48.0
25%       48.0
50%       48.0
75%       48.0
max       48.0
dtype: float64

Parcelas con número de meses distinto de 48: 0

Mínimo de meses por parcela: 48
Máximo de meses por parcela: 48


In [81]:
cobertura_mensual = (
    chirps_parcel_month
    .groupby(["year", "month"])
    .size()
    .reset_index(name="n_parcelas")
)

print(cobertura_mensual)

print(
    "\nMeses con != 197 parcelas:",
    (cobertura_mensual["n_parcelas"] != 197).sum()
)

    year  month  n_parcelas
0   2022      1         197
1   2022      2         197
2   2022      3         197
3   2022      4         197
4   2022      5         197
5   2022      6         197
6   2022      7         197
7   2022      8         197
8   2022      9         197
9   2022     10         197
10  2022     11         197
11  2022     12         197
12  2023      1         197
13  2023      2         197
14  2023      3         197
15  2023      4         197
16  2023      5         197
17  2023      6         197
18  2023      7         197
19  2023      8         197
20  2023      9         197
21  2023     10         197
22  2023     11         197
23  2023     12         197
24  2024      1         197
25  2024      2         197
26  2024      3         197
27  2024      4         197
28  2024      5         197
29  2024      6         197
30  2024      7         197
31  2024      8         197
32  2024      9         197
33  2024     10         197
34  2024     11     

In [82]:
fechas = (
    chirps_parcel_month[["year", "month"]]
    .drop_duplicates()
    .sort_values(["year", "month"])
    .reset_index(drop=True)
)

fechas["periodo"] = pd.to_datetime(
    fechas["year"].astype(str)
    + "-"
    + fechas["month"].astype(str)
    + "-01"
)

esperadas = pd.date_range(
    start="2022-01-01",
    end="2025-12-01",
    freq="MS"
)

print("Periodos encontrados:", len(fechas))
print("Periodos esperados:", len(esperadas))

print(
    "Periodos faltantes:",
    esperadas.difference(fechas["periodo"]).tolist()
)

print(
    "Periodos inesperados:",
    fechas.loc[
        ~fechas["periodo"].isin(esperadas),
        "periodo"
    ].tolist()
)

Periodos encontrados: 48
Periodos esperados: 48
Periodos faltantes: []
Periodos inesperados: []


In [83]:
chirps_parcel_month["area_weighted_mm"].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99
    ]
)

count    9456.000000
mean       69.513798
std        71.652590
min         0.000000
1%          0.000000
5%          5.756959
25%        14.562220
50%        38.220311
75%       111.911430
95%       227.131850
99%       302.370544
max       392.024017
Name: area_weighted_mm, dtype: float64

In [84]:
print(
    "Valores negativos:",
    (chirps_parcel_month["area_weighted_mm"] < 0).sum()
)

print(
    "Valores exactamente cero:",
    (chirps_parcel_month["area_weighted_mm"] == 0).sum()
)

print(
    "Valores > 500 mm:",
    (chirps_parcel_month["area_weighted_mm"] > 500).sum()
)

print(
    "Valores > 800 mm:",
    (chirps_parcel_month["area_weighted_mm"] > 800).sum()
)

print(
    "Valores > 1000 mm:",
    (chirps_parcel_month["area_weighted_mm"] > 1000).sum()
)

Valores negativos: 0
Valores exactamente cero: 222
Valores > 500 mm: 0
Valores > 800 mm: 0
Valores > 1000 mm: 0


In [85]:
top_precipitacion = (
    chirps_parcel_month[
        [
            "ID_POLIGON",
            "year",
            "month",
            "area_weighted_mm",
            "n_pixeles"
        ]
    ]
    .sort_values(
        "area_weighted_mm",
        ascending=False
    )
    .head(20)
)

top_precipitacion

,ID_POLIGON,year,month,area_weighted_mm,n_pixeles
4688,AGC_098,2024,9,392.024017,1
32,AGC_001,2024,9,392.024017,1
1376,AGC_029,2024,9,392.024017,1
5840,AGC_122,2024,9,387.149372,2
2912,AGC_061,2024,9,386.447479,1
3536,AGC_074,2024,9,386.447479,1
2910,AGC_061,2024,7,382.074951,1
3534,AGC_074,2024,7,382.074951,1
5838,AGC_122,2024,7,378.879325,2
5657,AGC_118,2025,6,366.062653,1


In [ ]:
ceros_por_mes = (
    chirps_parcel_month
    .assign(
        es_cero=lambda df:
        df["area_weighted_mm"] == 0
    )
    .groupby(["year", "month"])
    .agg(
        n_parcelas=("ID_POLIGON", "size"),
        n_ceros=("es_cero", "sum"),
        precip_media=("area_weighted_mm", "mean"),
        precip_mediana=("area_weighted_mm", "median")
    )
    .reset_index()
)

ceros_por_mes

,year,month,n_parcelas,n_ceros,precip_media,precip_mediana
0,2022,1,197,0,18.525858,20.171183
1,2022,2,197,0,10.318104,10.704084
2,2022,3,197,0,9.005980,10.546624
3,2022,4,197,0,30.834154,31.004378
4,2022,5,197,0,23.582298,19.288461
5,2022,6,197,0,82.155595,79.641838
6,2022,7,197,0,99.171222,79.025581
7,2022,8,197,0,146.792232,151.459026
8,2022,9,197,0,79.200266,76.690170
9,2022,10,197,0,69.364988,68.684456


In [87]:
ceros_por_mes[
    ceros_por_mes["n_ceros"] > 0
]

,year,month,n_parcelas,n_ceros,precip_media,precip_mediana
12,2023,1,197,48,4.981275,6.341150
13,2023,2,197,167,0.644261,0.000000
26,2024,3,197,7,4.875379,5.756959


In [88]:
resumen_mensual = (
    chirps_parcel_month
    .groupby("month")["area_weighted_mm"]
    .agg(
        media="mean",
        mediana="median",
        minimo="min",
        maximo="max",
        desviacion="std",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

resumen_mensual

,month,media,mediana,minimo,maximo,desviacion,p25,p75
0,1,12.417198,11.483009,0.000000,32.748020,6.641047,6.900209,17.910559
1,2,15.499972,11.878438,0.000000,61.019646,13.771457,6.116073,25.329439
2,3,13.006343,9.132049,0.000000,54.688824,10.697334,6.254598,13.457248
3,4,23.970043,24.446480,8.309810,48.421150,10.188376,14.156239,32.216301
4,5,75.483473,67.792496,13.668028,232.715652,49.725916,36.088875,106.401596
5,6,129.170396,108.450127,19.379429,366.062653,88.142048,55.141805,195.057495
6,7,155.635743,129.912888,61.024146,382.074951,82.309025,83.674164,214.984182
7,8,140.136608,136.458511,84.784164,243.131470,23.653856,121.938774,155.979172
8,9,129.882506,116.888588,31.889704,392.024017,78.881951,57.309681,188.081085
9,10,88.768367,77.097771,38.594769,167.598251,32.753993,63.080746,117.003632


In [89]:
import rasterio
import numpy as np

meses_ceros = [
    (2023, 1),
    (2023, 2),
    (2024, 3)
]

for year, month in meses_ceros:

    raster_path = (
        PRECIP_DIR
        / str(year)
        / f"PREC_{year}_{month:02d}.tif"
    )

    with rasterio.open(raster_path) as src:
        data = src.read(1)
        nodata = src.nodata

        valid = data[data != nodata]

        print(f"\n{year}-{month:02d}")
        print("  Raster válido min:", valid.min())
        print("  Raster válido max:", valid.max())
        print("  Píxeles exactamente 0:", np.sum(valid == 0))
        print(
            "  Total píxeles válidos:",
            len(valid)
        )
        print(
            "  % píxeles exactamente 0:",
            np.mean(valid == 0) * 100
        )


2023-01
  Raster válido min: 0.0
  Raster válido max: 50.13238
  Píxeles exactamente 0: 983
  Total píxeles válidos: 2032
  % píxeles exactamente 0: 48.375984251968504

2023-02
  Raster válido min: 0.0
  Raster válido max: 57.64569
  Píxeles exactamente 0: 1194
  Total píxeles válidos: 2032
  % píxeles exactamente 0: 58.75984251968504

2024-03
  Raster válido min: 0.0
  Raster válido max: 60.782063
  Píxeles exactamente 0: 662
  Total píxeles válidos: 2032
  % píxeles exactamente 0: 32.57874015748031


In [90]:
CHIRPS_INTERIM_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "climate"
)

CHIRPS_INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(CHIRPS_INTERIM_DIR)

c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\interim\climate


In [91]:
chirps_output_path = (
    CHIRPS_INTERIM_DIR
    / "chirps_parcel_month_area_weighted.csv"
)

chirps_parcel_month.to_csv(
    chirps_output_path,
    index=False
)

print(
    "Guardado en:",
    chirps_output_path
)

print(
    "Tamaño:",
    chirps_output_path.stat().st_size,
    "bytes"
)

Guardado en: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\interim\climate\chirps_parcel_month_area_weighted.csv
Tamaño: 833480 bytes


In [92]:
chirps_check = pd.read_csv(
    chirps_output_path
)

print("Shape:", chirps_check.shape)

print(
    "Duplicados parcela-mes:",
    chirps_check.duplicated(
        subset=["ID_POLIGON", "year", "month"]
    ).sum()
)

print(
    "Nulos:",
    chirps_check.isna().sum().sum()
)

print(
    "Parcelas:",
    chirps_check["ID_POLIGON"].nunique()
)

print(
    "Periodos:",
    chirps_check[["year", "month"]]
    .drop_duplicates()
    .shape[0]
)

Shape: (9456, 8)
Duplicados parcela-mes: 0
Nulos: 0
Parcelas: 197
Periodos: 48
